In [17]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

def setup_driver():
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920x1080")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
    
    # Automatically download and install ChromeDriver
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    return driver

def scrape_ipl_auction_data(driver):
    try:
        driver.get("https://www.iplt20.com/auction/2025")
        
        # Wait for page to load completely
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CLASS_NAME, "auction-tbl"))
        )
        time.sleep(3)  # Additional wait for dynamic content
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        team_sections = soup.find_all('div', class_='ih-pcard-sec')
        
        all_players = []
        
        for section in team_sections:
            team_header = section.find_previous('h2')
            if not team_header:
                continue
                
            team_name = team_header.get_text(strip=True)
            team_name = ' '.join(team_name.split()[1:])  # Remove numbering
            
            table = section.find('table', class_='auction-tbl')
            if not table:
                continue
                
            tbody = table.find('tbody', id='pointsdata')
            if not tbody:
                continue
                
            for row in tbody.find_all('tr'):
                cols = row.find_all('td')
                if len(cols) < 5:
                    continue
                
                try:
                    player_data = {
                        'Team': team_name,
                        'Player': cols[1].get_text(strip=True),
                        'Base Price': cols[2].get_text(strip=True),
                        'Winning Bid': cols[3].get_text(strip=True),
                        'Capped/Uncapped': cols[4].get_text(strip=True),
                        'Overseas': 'Yes' if cols[1].find('img', src=lambda x: x and 'overseas.svg' in x) else 'No',
                        'RTM': 'Yes' if len(cols) > 5 and cols[5].find('img', src=lambda x: x and 'rtm.svg' in x) else 'No'
                    }
                    all_players.append(player_data)
                except Exception as e:
                    print(f"Skipping row due to error: {str(e)}")
                    continue
        
        return pd.DataFrame(all_players) if all_players else None
    
    except Exception as e:
        print(f"Scraping error: {str(e)}")
        return None

def main():
    print("Starting scraping process...")
    driver = None
    
    try:
        driver = setup_driver()
        auction_data = scrape_ipl_auction_data(driver)
        
        if auction_data is not None:
            auction_data.to_csv('ipl_2025_auction_data.csv', index=False)
            print("Successfully saved data to 'ipl_2025_auction_data.csv'")
            print("\nSample data:")
            print(auction_data.head())
        else:
            print("No data was scraped. The website structure may have changed.")
            
    except Exception as e:
        print(f"Main error: {str(e)}")
    finally:
        if driver:
            driver.quit()

if __name__ == "__main__":
    main()

Starting scraping process...
Successfully saved data to 'ipl_2025_auction_data.csv'

Sample data:
          Team                Player    Base Price    Winning Bid Capped/Uncapped Overseas  RTM
0  Super Kings            Noor Ahmad  ₹2,00,00,000  ₹10,00,00,000          Capped      Yes   No
1  Super Kings  Ravichandaran Ashwin  ₹2,00,00,000   ₹9,75,00,000          Capped       No   No
2  Super Kings          Devon Conway  ₹2,00,00,000   ₹6,25,00,000          Capped      Yes   No
3  Super Kings    Syed Khaleel Ahmed  ₹2,00,00,000   ₹4,80,00,000          Capped       No   No
4  Super Kings       Rachin Ravindra  ₹1,50,00,000   ₹4,00,00,000          Capped      Yes  Yes


In [16]:
pip install webdriver-manager

  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
  Using cached python_dotenv-1.1.0-py3-none-any.whl.metadata (24 kB)
Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl (27 kB)
Using cached python_dotenv-1.1.0-py3-none-any.whl (20 kB)
Note: you may need to restart the kernel to use updated packages.
